
В данном домашнем задании вам необходимо обучить языковую модель (не более 1B параметров) решать примеры на сложение как можно более длинных чисел.

**Ожидаемый результат.** Необходимо предоставить код, а также технический отчет, содержащий описание метода и используемых данных, оценку качества (точность сложения).

**Пояснения:**

1. Можно использовать любые предобученные модели, можно их файнтюнить, обучать с нуля, адаптировать любым другим способом или брать как есть. Главное, чтобы все использованные вами идеи, код или веса моделей были описаны в приложенном отчете со ссылкой на источник.

2. Мы ожидаем, что ваш код принимает на вход два числа (в виде строк их десятичной записи) и выдает ответ в любом человекочитаемом виде. Однако, если ваша модель работает с входом в виде предложения на естественном языке, это тоже нормально, главное, чтобы это было описано в отчете.

3. Можно оценить качество работы алгоритма, посчитав accuracy на случайных множествах чисел разной длины. Если вам кажется более подходящей другая метрика, мы примем ваше решение. Опишите вашу метрику и аргументируйте выбор в отчете.

**Подсказка:** в качестве ориентира можете использовать следующий репозиторий (https://github.com/liutiedong/goat). В нем реализована сборка датасета для обучения и самообучения.

### Разбаловка

- [2 балла] Сбор датасета для обучения.  
- [3 балла] Реализация скрипта модели.
- [3 балла] Создание обученной модели с качеством (100% на числах длины < 10).
- [2 баллов] Валидация модели.

Для валидации можно использовать и готовую языковую модель без получения баллов за третий пункт.

## imports

In [1]:
import os
import random
import numpy as np
from datasets import load_dataset
from unsloth import FastModel, FastLanguageModel, is_bfloat16_supported
import torch
from trl import SFTTrainer
from transformers import TrainingArguments
from dotenv import load_dotenv, dotenv_values
import wandb

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

max_seq_length = 2048

load_dotenv()
hf_token  = dotenv_values().get("HF_TOKEN")
wandb_key = dotenv_values().get("WANDB_API_KEY")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
# Возьмем готовый датасет. Отфильтруем по операции сложения.
goat_ds = load_dataset("tiedong/goat")

In [3]:
goat_ds["train"][0]

{'instruction': '530991+6051993',
 'output': '530991 + 6051993 = 6582984',
 'answer': '6582984',
 'input': '530991 + 6051993'}

In [4]:
# Фильтруем датасет по условию: в поле input должен быть символ '+'
ds = goat_ds['train'].filter(lambda x: '+' in x['input'])


In [5]:
ds[-123]

{'instruction': 'Determine 33473 plus 432195868387250',
 'output': '33473 + 432195868387250 = 432195868420723',
 'answer': '432195868420723',
 'input': '33473 + 432195868387250'}

In [6]:
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs = examples["input"]
    outputs = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        if input_text.strip():
            user_message = f"{instruction}\n\n{input_text}"
        else:
            user_message = instruction
        messages = [
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": output},
        ]
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
            enable_thinking=False
        )
        texts.append(text)
    return {"text": texts}


In [7]:
ds = ds.map(formatting_prompts_func, batched=True, num_proc=10)

Map (num_proc=10):   0%|          | 0/304000 [00:00<?, ? examples/s]

NameError: name 'tokenizer' is not defined

## Model

In [ ]:
model_name = "unsloth/Qwen3-0.6B"
# model, tokenizer = FastModel.from_pretrained(
pt_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    # max_seq_length: Controls context length. While Qwen3 supports 40960, we 
    # recommend 2048 for testing. Unsloth enables 8× longer context fine-tuning
    max_seq_length=max_seq_length,    
    dtype=None,
    load_in_4bit=True,      # 4 bit quantization to reduce memory
    load_in_8bit=False,     # A bit more accurate, uses 2x memory
    full_finetuning=False,  
    token =hf_token,
)

==((====))==  Unsloth 2025.11.1: Fast Qwen3 patching. Transformers: 4.57.1.
   \\   /|    NVIDIA GeForce RTX 3080. Num GPUs = 1. Max memory: 9.64 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [ ]:
print(f"is_bfloat16_supported: {is_bfloat16_supported()}")
model = FastLanguageModel.get_peft_model(
    pt_model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
)



## Training


In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,
        # fp16=not is_bfloat16_supported(),
        fp16=False,
        # bf16=is_bfloat16_supported(),
        bf16=True,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=SEED,  # Make sure to set this!
        output_dir="outputs",
        report_to="wandb",
        # formatting_prompts_func=formatting_prompts_func,
    ),
)

In [ ]:
# Unsloth был скомпилирован без wandb. Чистим его кэш, он будет сбилден заного.

#!rm -rf /home/rin/project/ml-modern/homework/hw05/unsloth_compiled_cache

In [ ]:
wandb.login(key=wandb_key)
wandb.init()

trainer.train()


wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


train/epoch,▁▁▁▁▁▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
train/global_step,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
train/grad_norm,█▇▇▄▃▃▃▃▃▂▂▂▂▂▂▂▁▂▂▄▂▂▁▂▂▂▂▂▂▂▁▂▂▁▂▁▃▁▁▁
train/learning_rate,▂▄▅▇███▇▇▇▇▇▇▆▆▆▆▆▆▅▅▅▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▁▁
train/loss,▇█▇▆▅▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▁▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/epoch,0.00158
train/global_step,60
train/grad_norm,0.51086
train/learning_rate,0.0
train/loss,0.6387


The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 304,000 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 10,092,544 of 606,142,464 (1.67% trained)


Step,Training Loss
1,0.728900
2,0.657700
3,0.697400
4,0.711600
5,0.733700
6,0.732400
7,0.721200
8,0.732000
9,0.684000
10,0.733300


NameError: name 'wandb' is not defined

### Описание вашего решения

[TODO]

1. Возьмем готовый датасет. Отфильтруем по операции сложения.
2. Возьмем модель Qwen3-0.6B, подходит по параметрам, свежая модель. 
    И опыт FT Qwen3 мне пригодиться в ближайшее время.
3. Проведем FT с помощью фреймворка unsloth.
    Ни разу им не пользовался, но часто про него слышу. 
4. Провалидируем модель


### *[1 балл] Дополнительное задание

Реализовать хостинг вашей модели на gradio.

### Выводы

В этом задании вы научились решать арифметические действия с помощью языковой модели.